In [47]:
import pandas as pd
import re
import ast
from html import escape
from IPython.display import display, HTML

In [48]:
# read the CSV file into a DataFrame
df = pd.read_csv('subtitles_ai_related.csv', index_col=0)

In [49]:
old_df = df.copy()
# delete rows with mention of "songfestical" (false positives)
df = df[~df['text_clean'].str.contains('songfestival', case=False, na=False)]
# show how many rows dropped
dropped = len(old_df) - len(df)
print(f"Dropped {dropped} rows with 'songfestival' mentions.")


Dropped 42 rows with 'songfestival' mentions.


In [50]:
import ast

def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

df['matched_keywords_all'] = df.apply(
    lambda row: to_list(row['company_hits']) + to_list(row['matched_keywords_all']),
    axis=1
)
df['matched_keywords_all'].iloc[0]

['facebook', 'google', 'twitter', 'algoritme']

In [52]:
#drop rows
df = df.drop(columns=['company_hits', 'matched_keywords_title', 'matched_keywords_body', 'n_hits_title_total', 'n_hits_body_total'])

In [53]:
df

,filename,program,year,text_clean,ai_related,matched_keywords_all
47,"2016-11-16-19,00-1",DE WERELD DRAAIT DOOR,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"[facebook, google, twitter, algoritme]"
52,"2016-02-08-23,04-1",JINEK,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,[drones]
56,GOEDEMORGEN_N-WON02434179,Goedemorgen Nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"[ai, kunstmatige intelligentie]"
74,GOEDEMORGEN_N-WON02298553,Goedemorgen Nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"[twitter, kunstmatige intelligentie]"
153,GOEDEMORGEN_N-WON02108661,Goedemorgen Nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,[drone]
...,...,...,...,...,...,...
9928,"2017-09-18-19,01-1",DE WERELD DRAAIT DOOR,2017,"888\nGeerte Piening, Bahar Goodarzi, Jeroen Wo...",yes,"[facebook, twitter, robot]"
9936,GOEDEMORGEN_N-WON02358587,Goedemorgen Nederland,2022,"888\nGoedemorgen Nederland, welkom terug bij W...",yes,"[drone, drones]"
9941,"2018-04-27-08,38-1",Goedemorgen Nederland,2018,"888\nGoedemorgen Nederland, het is vandaag vri...",yes,"[twitter, drones]"
9946,GOEDEMORGEN_N-WON02320372,Goedemorgen Nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"[ai, drones]"


In [54]:


def inspect_ai_related(df, program, body='relevant_section',  num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """
    
    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'program', body}.issubset(df.columns):
        missing = {'program', body} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # # --- sample ---
    # n = min(num_samples, len(df))
    # samples = df.sample(n=n, random_state=random_state)

    # --- sample ---
    
    if program == "all":
        pool = df
    else:
        pool = df[df['program'] == program]
       
    
      

    n = min(num_samples, len(pool))
    samples = pool.sample(n=n, random_state=random_state)


    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        
        ai_val = row['ai_related']
        title = row['program']
        body_text = row[body]  # ✅ 'body' parameter stays intact

         # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body_text, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples
    

In [55]:
import re
import ast
import pandas as pd

def _norm_matched_keywords(val):
    """
    Normalize matched_keywords_all into a list[str].
    Accepts: list/set/tuple, stringified list (e.g. "['ai','ml']"),
             or comma-separated string ("ai, machine learning").
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, set, tuple)):
        kws = list(val)
    elif isinstance(val, str):
        s = val.strip()
        if not s:
            return []
        # Try to parse stringified list
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, set, tuple)):
                kws = list(parsed)
            else:
                # fallback: treat as comma-separated
                kws = [x.strip() for x in s.split(",") if x.strip()]
        except Exception:
            kws = [x.strip() for x in s.split(",") if x.strip()]
    else:
        return []

    # clean, dedupe, keep phrases
    out = []
    seen = set()
    for k in kws:
        if not isinstance(k, str):
            continue
        k2 = k.strip()
        if not k2:
            continue
        kl = k2.lower()
        if kl not in seen:
            seen.add(kl)
            out.append(k2)
    return out

def _extract_windows(text, keywords, window=5, max_snippets=5):
    """
    Build a plain-text section composed of ±window words around each keyword hit.
    Returns: (section_text, spans, n_matches)
    """
    if not isinstance(text, str) or not text.strip():
        return "", [], 0
    keywords = _norm_matched_keywords(keywords)
    if not keywords:
        return "", [], 0

    # Safer "word boundary" for phrases: (?<!\w) ... (?!\w)
    # (handles multi-word phrases and punctuation better than \b ... \b)
    alts = [re.escape(k) for k in keywords]
    pattern = r'(?<!\w)(?:' + '|'.join(alts) + r')(?!\w)'
    flags = re.IGNORECASE

    matches = list(re.finditer(pattern, text, flags))
    if not matches:
        return "", [], 0

    # Word spans to take ±window words
    word_spans = [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]

    def word_index_for_pos(pos):
        # small linear scan is fine for typical lengths
        for i, (s, e) in enumerate(word_spans):
            if s <= pos < e:
                return i
        # if match starts on whitespace, snap to next word
        for i, (s, e) in enumerate(word_spans):
            if pos < e:
                return i
        return len(word_spans) - 1 if word_spans else 0

    spans = []
    for m in matches:
        wi = word_index_for_pos(m.start())
        start_w = max(0, wi - window)
        end_w   = min(len(word_spans), wi + window + 1)
        start_c = word_spans[start_w][0]
        end_c   = word_spans[end_w - 1][1]

        if spans and start_c <= spans[-1][1]:
            spans[-1] = (spans[-1][0], max(spans[-1][1], end_c))
        else:
            spans.append((start_c, end_c))

    spans = spans[:max_snippets]

    parts = []
    for i, (s, e) in enumerate(spans):
        prefix = '...' if (i == 0 and s > 0) else ''
        suffix = '...' if (i == len(spans)-1 and e < len(text)) else ''
        parts.append(prefix + text[s:e].strip() + suffix)

    section_text = ' '.join(parts)
    return section_text, spans, len(matches)

def add_relevant_section(df, body_col='body', matched_col='matched_keywords_all',
                         window=5, max_snippets=5):
    """
    Adds:
      - relevant_section (plain text)
      - relevant_section_spans (char spans)
      - n_keyword_matches
      - relevant_len_chars
      - relevant_len_words
    """
    results = df.apply(
        lambda r: _extract_windows(
            str(r[body_col]) if pd.notna(r.get(body_col)) else "",
            r.get(matched_col),
            window,
            max_snippets
        ),
        axis=1
    )
    df['relevant_section']       = results.map(lambda t: t[0])
    df['relevant_section_spans'] = results.map(lambda t: t[1])
    df['n_keyword_matches']      = results.map(lambda t: t[2])
    df['relevant_len_chars']     = df['relevant_section'].str.len()
    df['relevant_len_words']     = df['relevant_section'].str.count(r'\b\w+\b')
    return df


In [56]:
df_relevant = add_relevant_section(df, body_col='text_clean', matched_col='matched_keywords_all', window=50)
df_relevant

,filename,program,year,text_clean,ai_related,matched_keywords_all,relevant_section,relevant_section_spans,n_keyword_matches,relevant_len_chars,relevant_len_words
47,"2016-11-16-19,00-1",DE WERELD DRAAIT DOOR,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"[facebook, google, twitter, algoritme]",...uit Amerika.\nVanochtend geland.\nHet Ameri...,"[(352, 1482), (6045, 6620), (21504, 22090)]",6,2299,392
52,"2016-02-08-23,04-1",JINEK,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,[drones],...over.\nEn onze tennisdames verrasten de wer...,"[(17898, 18694)]",2,802,131
56,GOEDEMORGEN_N-WON02434179,Goedemorgen Nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"[ai, kunstmatige intelligentie]",...en de neus van de politie.\nWeet u hoe een ...,"[(6024, 7855)]",5,1837,294
74,GOEDEMORGEN_N-WON02298553,Goedemorgen Nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"[twitter, kunstmatige intelligentie]",...zegt de Britse minister van buitenlandse za...,"[(704, 1351), (9419, 10034)]",2,1269,204
153,GOEDEMORGEN_N-WON02108661,Goedemorgen Nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,[drone],...er aardig uit.\nIs het een quarantainecoup?...,"[(4852, 5768)]",3,922,166
...,...,...,...,...,...,...,...,...,...,...,...
9928,"2017-09-18-19,01-1",DE WERELD DRAAIT DOOR,2017,"888\nGeerte Piening, Bahar Goodarzi, Jeroen Wo...",yes,"[facebook, twitter, robot]",...de Staten-Generaal' en niet 'mongool'.\nLev...,"[(2686, 3332), (10448, 10998), (21062, 21630)]",4,1772,326
9936,GOEDEMORGEN_N-WON02358587,Goedemorgen Nederland,2022,"888\nGoedemorgen Nederland, welkom terug bij W...",yes,"[drone, drones]",...prijsgeven...\nmaar wil aan buitenlandse bo...,"[(5358, 6309)]",4,957,153
9941,"2018-04-27-08,38-1",Goedemorgen Nederland,2018,"888\nGoedemorgen Nederland, het is vandaag vri...",yes,"[twitter, drones]",...Zo gaat zij.\nHeel vrolijk.\nAnneke Nieuwen...,"[(4217, 4778), (9012, 9573)]",2,1129,203
9946,GOEDEMORGEN_N-WON02320372,Goedemorgen Nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"[ai, drones]",...weten het niet.\nMaar zij willen iets doen....,"[(2787, 3368), (4133, 4709)]",3,1164,204


In [57]:
df = df.drop(columns=['relevant_section_spans', 'n_keyword_matches', 'relevant_len_chars', 'relevant_len_words'])

In [58]:
# lowercase all programs for easier filtering
df['program'] = df['program'].str.lower()

In [59]:
df['program'].unique()

<StringArray>
['de wereld draait door',                 'jinek', 'goedemorgen nederland',
                   'eva',          'de vooravond',                  'pauw',
              'bar laat',       'café kockelmann',                     'm']
Length: 9, dtype: str

In [ ]:
# # filter for certain keywords
# keywords_of_interest = ['programma']
# pattern = '|'.join([re.escape(k) for k in keywords_of_interest])
# df_filtered_keywords = df[df['matched_keywords_all'].apply(lambda kws: any(re.search(pattern, str(k), re.IGNORECASE) for k in to_list(kws)))]

In [ ]:
# # search for "facebook" in the relevant_section as well (in case it was missed in the keyword matching)
# df_filtered_keywords = df[df['relevant_section'].str.contains('programma', case=False, na=False)]
# # add twitter to keywords for highlighting
# df_filtered_keywords['matched_keywords_all'] = df_filtered_keywords['matched_keywords_all'].apply(lambda kws: to_list(kws) + ['programma'])


In [61]:
# df_filtered_keywords.info()

In [64]:
# program = "all" to display from all programs, or specify a program like "jinek"
inspect_ai_related(df, body = 'relevant_section', num_samples=1, random_state=42, program='all')

Displayed 1 random articles (with row-specific matched keywords highlighted).


,filename,program,year,text_clean,ai_related,matched_keywords_all,relevant_section
3509,GOEDEMORGEN_N-WON02399764,goedemorgen nederland,2023,Goedemorgen Nederland.\nKleine verhuurders mak...,yes,"[ai, chatgpt, kunstmatige intelligentie]",...en die kreeg te horen:\nuw huurcontract sto...


In [264]:
df

,filename,program,year,text_clean,ai_related,matched_keywords_all,relevant_section
47,"2016-11-16-19,00-1",de wereld draait door,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"[facebook, google, algoritme]",...uit Amerika.\nVanochtend geland.\nHet Ameri...
52,"2016-02-08-23,04-1",jinek,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,[drones],...over.\nEn onze tennisdames verrasten de wer...
56,GOEDEMORGEN_N-WON02434179,goedemorgen nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"[ai, kunstmatige intelligentie]",...en de neus van de politie.\nWeet u hoe een ...
153,GOEDEMORGEN_N-WON02108661,goedemorgen nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,[drone],...er aardig uit.\nIs het een quarantainecoup?...
189,GOEDEMORGEN_N-WON02301038,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"[facebook, drones]",...bereid om heel ver te gaan.\nHet bombardere...
...,...,...,...,...,...,...,...
9914,BAR_LAAT_____-WON02531286,bar laat,2024,"888\nGoedenavond.\nSuse van Kleef, Michiel Rom...",yes,"[google, ai]",...meer over de nieuwe misstanden bij het Arnh...
9928,"2017-09-18-19,01-1",de wereld draait door,2017,"888\nGeerte Piening, Bahar Goodarzi, Jeroen Wo...",yes,"[facebook, robot]",...de Staten-Generaal' en niet 'mongool'.\nLev...
9936,GOEDEMORGEN_N-WON02358587,goedemorgen nederland,2022,"888\nGoedemorgen Nederland, welkom terug bij W...",yes,"[drone, drones]",...prijsgeven...\nmaar wil aan buitenlandse bo...
9946,GOEDEMORGEN_N-WON02320372,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"[ai, drones]",...weten het niet.\nMaar zij willen iets doen....


In [65]:
df.to_csv('subtitles_relevant_sections_50.csv')